# Gradient Boosting from scratch
Sections follow the lesson:
1. Regression tree helper (leaf values can be overwritten -> per-leaf step size gamma)
2. Worked example: targets [100,120,140,160], F0 = mean, residuals, one tree
3. Gradient view: pseudo-residuals = negative gradient of the loss (MSE, MAE, Huber, log-loss)
4. Squared-loss `GradientBoostingRegressor` and shrinkage (learning rate)
5. Hyperparameters: n_trees, depth, subsample (stochastic GB), early stopping
6. Log-loss `GradientBoostingClassifier` (F0 = log-odds, Newton leaf values)
7. Robust losses on outliers, and comparison with AdaBoost (exponential loss)

In [ ]:
import os
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. A small regression tree
Greedy CART with variance-reduction splits; leaf value = mean of targets. `apply` returns the leaf
id of each row, so we can later replace each leaf value by the loss-specific optimal step `gamma`.

In [ ]:
class RegTree:
    def __init__(self, max_depth=3, min_samples_leaf=1):
        self.max_depth, self.msl = max_depth, min_samples_leaf

    def fit(self, X, r):
        self.feat, self.thr, self.left, self.right, self.value = [], [], [], [], []
        self._grow(X, r.astype(float), 0)
        for k in ("feat", "thr", "left", "right", "value"):
            setattr(self, k, np.array(getattr(self, k)))
        return self

    def _grow(self, X, r, depth):
        node = len(self.feat)
        for lst, v in ((self.feat, -1), (self.thr, 0.0), (self.left, -1), (self.right, -1), (self.value, r.mean())):
            lst.append(v)
        if depth >= self.max_depth or len(r) < 2 * self.msl or np.ptp(r) == 0:
            return node
        j, t = self._best_split(X, r)
        if j is None:
            return node
        m = X[:, j] <= t
        self.feat[node], self.thr[node] = j, t
        self.left[node] = self._grow(X[m], r[m], depth + 1)
        self.right[node] = self._grow(X[~m], r[~m], depth + 1)
        return node

    def _best_split(self, X, r):
        n, d = X.shape
        S, best_gain, best = r.sum(), 1e-12, (None, None)
        for j in range(d):
            o = np.argsort(X[:, j], kind="stable")
            xs, cs = X[o, j], np.cumsum(r[o])[:-1]
            k = np.arange(1, n)
            valid = (xs[:-1] < xs[1:]) & (k >= self.msl) & (n - k >= self.msl)
            if not valid.any():
                continue
            gain = np.where(valid, cs**2 / k + (S - cs) ** 2 / (n - k) - S**2 / n, -np.inf)
            i = int(np.argmax(gain))
            if gain[i] > best_gain:
                best_gain, best = gain[i], (j, 0.5 * (xs[i] + xs[i + 1]))
        return best

    def apply(self, X):
        node = np.zeros(len(X), dtype=int)
        active = self.left[node] != -1
        while active.any():
            idx = np.where(active)[0]
            nd = node[idx]
            node[idx] = np.where(X[idx, self.feat[nd]] <= self.thr[nd], self.left[nd], self.right[nd])
            active = self.left[node] != -1
        return node

    def predict(self, X):
        return self.value[self.apply(X)]

## 2. Worked example (from the lesson)
Targets `[100,120,140,160]`, `F0 = mean = 130`, residuals `[-30,-10,10,30]`.
We fit a stump on a feature that orders the samples (e.g. size) and add it with learning rate `eta`.

In [ ]:
x4 = np.array([[1.0], [2.0], [3.0], [4.0]])
y4 = np.array([100., 120., 140., 160.])
F = np.full(4, y4.mean())
print("F0 =", F[0], "| residuals:", y4 - F)
eta = 0.5
for t in range(1, 4):
    r = y4 - F
    tree = RegTree(max_depth=1).fit(x4, r)
    F = F + eta * tree.predict(x4)
    print("round %d: tree output %s -> F = %s | MSE %.1f" % (t, np.round(tree.predict(x4), 1), np.round(F, 1), ((y4 - F) ** 2).mean()))

# the optimal MSE step size gamma for a tree on residuals is exactly 1 (the leaf mean)
r = y4 - y4.mean(); h = RegTree(max_depth=1).fit(x4, r).predict(x4)
gam = np.linspace(0, 2, 201)
sse = [((y4 - (y4.mean() + g * h)) ** 2).sum() for g in gam]
print("line-search gamma for MSE: %.2f (leaf means already absorb the step)" % gam[int(np.argmin(sse))])
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(gam, sse); ax.axvline(gam[int(np.argmin(sse))], c="r", ls="--"); ax.set(xlabel="gamma", ylabel="SSE", title="Step-size line search (MSE)")
fig.tight_layout(); fig.savefig("plots/01_gamma_line_search.png"); plt.close(fig)

## 3. Gradient view: pseudo-residuals
`r_i = -dL/dF(x_i)` at the current prediction. Boosting = gradient descent in *function space*:
the tree is a (smoothed) fit of the negative gradient.

| loss | pseudo-residual |
|---|---|
| squared `(y-F)^2/2` | `y - F` |
| absolute `|y-F|` | `sign(y - F)` |
| Huber(delta) | `clip(y-F, -delta, delta)` |
| log-loss, y in {0,1} | `y - sigmoid(F)` |

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -35, 35)))

def numgrad(loss, y, F, eps=1e-6):
    return (loss(y, F + eps) - loss(y, F - eps)) / (2 * eps)

delta = 1.0
mse = lambda y, F: 0.5 * (y - F) ** 2
mae = lambda y, F: np.abs(y - F)
huber = lambda y, F: np.where(np.abs(y - F) <= delta, 0.5 * (y - F) ** 2, delta * (np.abs(y - F) - 0.5 * delta))
logl = lambda y, F: -(y * np.log(sigmoid(F)) + (1 - y) * np.log(1 - sigmoid(F)))
yy, FF = np.array([0.3, 2.5, -1.0, 4.0]), np.array([1.0, 0.2, 0.5, -1.5])
yb = np.array([1., 0., 1., 0.])
checks = [("MSE", mse, yy, yy - FF), ("MAE", mae, yy, np.sign(yy - FF)),
          ("Huber", huber, yy, np.clip(yy - FF, -delta, delta)), ("log-loss", logl, yb, yb - sigmoid(FF))]
for name, L, yv, res in checks:
    print("%-8s -numeric grad == pseudo-residual: %s" % (name, np.allclose(-numgrad(L, yv, FF), res, atol=1e-5)))

z = np.linspace(-4, 4, 300)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(z, 0.5 * z**2, label="squared"); ax[0].plot(z, np.abs(z), label="absolute")
ax[0].plot(z, np.where(np.abs(z) <= 1, 0.5 * z**2, np.abs(z) - 0.5), label="Huber"); ax[0].set(title="Losses vs residual", xlabel="y - F", ylim=(0, 6)); ax[0].legend()
ax[1].plot(z, z, label="squared"); ax[1].plot(z, np.sign(z), label="absolute"); ax[1].plot(z, np.clip(z, -1, 1), label="Huber")
ax[1].set(title="Pseudo-residual (negative gradient)", xlabel="y - F"); ax[1].legend()
fig.tight_layout(); fig.savefig("plots/02_losses_pseudo_residuals.png"); plt.close(fig)

## 4. Gradient boosting regressor (squared loss) and shrinkage
`F_0 = mean(y)`; loop: residuals -> tree -> `F += eta * tree`. Optional row subsampling
(stochastic gradient boosting). Supports squared / absolute / Huber losses; for non-MSE losses the
leaf values are replaced by the loss-optimal constant (median of residuals for MAE).

In [ ]:
class GBRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0, loss="squared", delta=1.0, min_samples_leaf=1, seed=0):
        self.msl = min_samples_leaf
        self.T, self.eta, self.depth, self.sub, self.loss, self.delta = n_estimators, learning_rate, max_depth, subsample, loss, delta
        self.rng = np.random.default_rng(seed)

    def _resid(self, y, F):
        r = y - F
        if self.loss == "squared": return r
        if self.loss == "absolute": return np.sign(r)
        return np.clip(r, -self.delta, self.delta)

    def fit(self, X, y, Xval=None, yval=None):
        n = len(y)
        self.F0 = y.mean() if self.loss == "squared" else np.median(y)
        F = np.full(n, self.F0)
        self.trees, self.train_loss, self.val_loss = [], [], []
        Fv = None if Xval is None else np.full(len(yval), self.F0)
        for _ in range(self.T):
            idx = np.arange(n) if self.sub >= 1 else self.rng.choice(n, int(self.sub * n), replace=False)
            tree = RegTree(self.depth, self.msl).fit(X[idx], self._resid(y[idx], F[idx]))
            if self.loss != "squared":             # leaf-wise optimal step
                leaf = tree.apply(X)
                for l in np.unique(leaf[idx]):
                    m = idx[leaf[idx] == l]
                    r = y[m] - F[m]
                    tree.value[l] = np.median(r) if self.loss == "absolute" else self._huber_leaf(r)
            F = F + self.eta * tree.predict(X)
            self.trees.append(tree)
            self.train_loss.append(((y - F) ** 2).mean())
            if Fv is not None:
                Fv = Fv + self.eta * tree.predict(Xval); self.val_loss.append(((yval - Fv) ** 2).mean())
        return self

    def _huber_leaf(self, r):
        med = np.median(r)
        return med + np.mean(np.clip(r - med, -self.delta, self.delta))

    def staged_predict(self, X):
        F = np.full(len(X), self.F0)
        for t in self.trees:
            F = F + self.eta * t.predict(X)
            yield F.copy()

    def predict(self, X):
        F = np.full(len(X), self.F0)
        for t in self.trees:
            F += self.eta * t.predict(X)
        return F


def make_reg(n, rng=rng, noise=0.25):
    X = rng.uniform(-3, 3, size=(n, 3))
    f = np.sin(2 * X[:, 0]) + 0.5 * X[:, 1] ** 2 * 0.4 + 0.5 * X[:, 0] * X[:, 2] * 0.3
    return X, f + noise * rng.normal(size=n), f

Xtr, ytr, _ = make_reg(400); Xva, yva, _ = make_reg(300); Xte, yte, fte = make_reg(1000)
gb = GBRegressor(200, 0.1, 3).fit(Xtr, ytr)
print("GB (200 trees, eta .1, depth 3): train MSE %.4f | test MSE %.4f | noise floor %.4f"
      % (gb.train_loss[-1], ((gb.predict(Xte) - yte) ** 2).mean(), 0.25**2))
print("training MSE decreases monotonically:", bool(np.all(np.diff(gb.train_loss) <= 1e-12)))

# 1-D visualisation of stagewise fitting
x1 = np.sort(rng.uniform(-3, 3, 150))[:, None]; y1 = np.sin(2 * x1[:, 0]) + 0.2 * rng.normal(size=150)
xg = np.linspace(-3, 3, 400)[:, None]
g1 = GBRegressor(100, 0.3, 2).fit(x1, y1)
stages = list(g1.staged_predict(xg))
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, m in zip(axes, (1, 3, 10, 100)):
    ax.scatter(x1, y1, s=6, c="gray"); ax.plot(xg, stages[m - 1], c="tab:red"); ax.set(title="%d tree(s)" % m)
fig.tight_layout(); fig.savefig("plots/03_stagewise_fit.png"); plt.close(fig)

# residuals shrink round by round
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
Fp = np.full(len(y1), y1.mean())
for a, t in zip(ax, (0, 1, 5)):
    Fm = Fp if t == 0 else np.array(list(GBRegressor(t, 0.3, 2).fit(x1, y1).staged_predict(x1))[-1])
    a.scatter(x1, y1 - Fm, s=6); a.axhline(0, c="k", lw=0.8); a.set(title="residuals after %d trees (std %.2f)" % (t, (y1 - Fm).std()))
fig.tight_layout(); fig.savefig("plots/04_residuals_shrink.png"); plt.close(fig)

### Learning rate (shrinkage)
Small `eta` takes small, cautious steps: it needs more trees but ends at a better validation error.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for eta in (1.0, 0.3, 0.1, 0.03):
    m = GBRegressor(400, eta, 3).fit(Xtr, ytr, Xva, yva)
    ax[0].plot(m.train_loss, label="eta=%.2f" % eta); ax[1].plot(m.val_loss, label="eta=%.2f" % eta)
    print("eta=%.2f: best val MSE %.4f at tree %3d | val MSE after 400 trees %.4f" % (eta, min(m.val_loss), int(np.argmin(m.val_loss)) + 1, m.val_loss[-1]))
ax[0].set(title="Train MSE", xlabel="trees", yscale="log"); ax[1].set(title="Validation MSE", xlabel="trees", ylim=(0.05, 0.6)); ax[1].legend()
fig.tight_layout(); fig.savefig("plots/05_learning_rate.png"); plt.close(fig)

## 5. Depth, subsample, early stopping
* depth 1 = additive model (no interactions); depth 3-6 captures interactions; very deep trees overfit.
* `subsample < 1` (stochastic GB) injects randomness: faster, lower variance.
* Early stopping: stop when validation loss has not improved for `patience` rounds.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for d in (1, 2, 3, 5, 8):
    m = GBRegressor(200, 0.1, d).fit(Xtr, ytr, Xva, yva)
    ax[0].plot(m.val_loss, label="depth %d" % d)
    print("depth %d: best val MSE %.4f at tree %3d | final train MSE %.4f" % (d, min(m.val_loss), int(np.argmin(m.val_loss)) + 1, m.train_loss[-1]))
ax[0].set(title="Validation MSE by depth", xlabel="trees", ylim=(0.05, 0.5)); ax[0].legend()
for s in (1.0, 0.5):
    vals = [min(GBRegressor(150, 0.1, 3, subsample=s, seed=k).fit(Xtr, ytr, Xva, yva).val_loss) for k in range(3)]
    print("subsample %.1f: best val MSE (mean of 3 seeds) %.4f" % (s, np.mean(vals)))
    m = GBRegressor(150, 0.1, 3, subsample=s, seed=0).fit(Xtr, ytr, Xva, yva); ax[1].plot(m.val_loss, label="subsample %.1f" % s)
ax[1].set(title="Stochastic gradient boosting", xlabel="trees", ylim=(0.05, 0.5)); ax[1].legend()
fig.tight_layout(); fig.savefig("plots/06_depth_subsample.png"); plt.close(fig)

m = GBRegressor(400, 0.1, 3).fit(Xtr, ytr, Xva, yva)
best, wait, stop = np.inf, 0, len(m.val_loss)
for i, v in enumerate(m.val_loss):
    if v < best - 1e-6: best, wait = v, 0
    else: wait += 1
    if wait >= 20: stop = i + 1; break
print("early stopping (patience 20) halts at tree %d | true best tree %d" % (stop, int(np.argmin(m.val_loss)) + 1))

## 6. Classification with log-loss
Model the log-odds `F`. `F0 = log(p/(1-p))`. Pseudo-residual `y - sigmoid(F)`. A tree is fit on it, then each
leaf gets the Newton step `gamma_leaf = sum(r) / sum(p(1-p))` (a one-step solution of the line search
`argmin_gamma sum L(y, F + gamma)`), and `F += eta * gamma_leaf`. Predicted probability = `sigmoid(F)`.

In [ ]:
class GBClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample=1.0, seed=0):
        self.T, self.eta, self.depth, self.sub = n_estimators, learning_rate, max_depth, subsample
        self.rng = np.random.default_rng(seed)

    def fit(self, X, y):
        n = len(y)
        p0 = y.mean()
        self.F0 = np.log(p0 / (1 - p0))
        F = np.full(n, self.F0)
        self.trees, self.train_loss = [], []
        for _ in range(self.T):
            p = sigmoid(F)
            r = y - p
            idx = np.arange(n) if self.sub >= 1 else self.rng.choice(n, int(self.sub * n), replace=False)
            tree = RegTree(self.depth).fit(X[idx], r[idx])
            leaf = tree.apply(X)
            for l in np.unique(leaf[idx]):
                m = idx[leaf[idx] == l]
                tree.value[l] = r[m].sum() / max((p[m] * (1 - p[m])).sum(), 1e-12)
            F = F + self.eta * tree.predict(X)
            self.trees.append(tree)
            p = np.clip(sigmoid(F), 1e-12, 1 - 1e-12)
            self.train_loss.append(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())
        return self

    def staged_decision(self, X):
        F = np.full(len(X), self.F0)
        for t in self.trees:
            F = F + self.eta * t.predict(X)
            yield F.copy()

    def decision_function(self, X):
        F = np.full(len(X), self.F0)
        for t in self.trees:
            F += self.eta * t.predict(X)
        return F

    def predict_proba(self, X):
        return sigmoid(self.decision_function(X))

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)


def make_ring(n, noise=0.0, rng=rng):
    X = rng.normal(size=(n, 2))
    y = (X[:, 0] ** 2 + X[:, 1] ** 2 > 1.4).astype(int)
    X = X + 0.15 * rng.normal(size=X.shape)
    return X, np.where(rng.random(n) < noise, 1 - y, y)

Xc, yc = make_ring(500, 0.05); Xct, yct = make_ring(2000)
gbc = GBClassifier(150, 0.1, 2).fit(Xc, yc)
print("GBClassifier: F0 (log-odds) %.3f | train acc %.3f test acc %.3f | train log-loss %.3f -> %.3f"
      % (gbc.F0, (gbc.predict(Xc) == yc).mean(), (gbc.predict(Xct) == yct).mean(), gbc.train_loss[0], gbc.train_loss[-1]))
print("log-loss decreases monotonically:", bool(np.all(np.diff(gbc.train_loss) < 1e-9)))

gx, gy = np.meshgrid(np.linspace(-3.2, 3.2, 150), np.linspace(-3.2, 3.2, 150)); G = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.7), sharex=True, sharey=True)
for ax, m in zip(axes, (1, 5, 20, 150)):
    Fm = list(gbc.staged_decision(G))[m - 1]
    cs = ax.contourf(gx, gy, sigmoid(Fm).reshape(gx.shape), levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.7)
    ax.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=5, edgecolor="k", linewidth=0.2); ax.set(title="%d tree(s): P(y=1)" % m)
fig.tight_layout(); fig.savefig("plots/07_classifier_boundary.png"); plt.close(fig)

# calibration: predicted probabilities vs observed frequency
pt = gbc.predict_proba(Xct); bins = np.linspace(0, 1, 11); b = np.clip(np.digitize(pt, bins) - 1, 0, 9)
obs = np.array([yct[b == k].mean() if (b == k).any() else np.nan for k in range(10)]); mid = 0.5 * (bins[:-1] + bins[1:])
fig, ax = plt.subplots(figsize=(4.8, 4)); ax.plot([0, 1], [0, 1], "k--"); ax.plot(mid, obs, "o-"); ax.set(xlabel="predicted P(y=1)", ylabel="observed frequency", title="Calibration (log-loss gives probabilities)")
fig.tight_layout(); fig.savefig("plots/08_calibration.png"); plt.close(fig)

## 7. Robust losses on outliers, and GB vs AdaBoost
Squared loss chases large residuals; absolute and Huber losses cap the pseudo-residual.
(A leaf holding a single outlier would still take its value, so robust GB also needs `min_samples_leaf > 1`.)

In [ ]:
x2 = np.sort(rng.uniform(-3, 3, 200))[:, None]; y2 = np.sin(x2[:, 0]) + 0.1 * rng.normal(size=200)
bad = y2.copy(); out = rng.choice(200, 10, replace=False); bad[out] += rng.choice([-1, 1], 10) * 8
xt = np.linspace(-3, 3, 400)[:, None]; truth = np.sin(xt[:, 0])
fig, ax = plt.subplots(figsize=(7, 4)); ax.scatter(x2, bad, s=6, c="gray")
for loss in ("squared", "absolute", "huber"):
    pred = GBRegressor(100, 0.1, 2, loss=loss, delta=0.3, min_samples_leaf=8).fit(x2, bad).predict(xt)
    ax.plot(xt, pred, label=loss); print("%-9s loss: RMSE vs true curve %.3f" % (loss, np.sqrt(((pred - truth) ** 2).mean())))
ax.plot(xt, truth, "k--", label="truth"); ax.set(title="10 outliers: squared loss is dragged away"); ax.legend(); ax.set_ylim(-4, 4)
fig.tight_layout(); fig.savefig("plots/09_robust_losses.png"); plt.close(fig)

# AdaBoost (exp loss, sample reweighting) as stump-based comparison on flipped labels: reuse GB log-loss with stumps
def ada_stump_boost(X, y01, T):
    y = 2 * y01 - 1; n = len(y); w = np.full(n, 1 / n); F = np.zeros(n); out = []
    for _ in range(T):
        best = (np.inf, 0, 0, 1)
        for j in range(X.shape[1]):
            o = np.argsort(X[:, j]); xs, ys, ws = X[o, j], y[o], w[o]
            pl = np.cumsum(ws * (ys == 1)); nl = np.cumsum(ws * (ys == -1)); e1 = pl + nl[-1] - nl
            ok = xs[:-1] < xs[1:]
            for pol, e in ((1, e1), (-1, 1 - e1)):
                ee = np.where(ok, e[:-1], np.inf); i = int(np.argmin(ee))
                if ee[i] < best[0]: best = (ee[i], j, 0.5 * (xs[i] + xs[i + 1]), pol)
        e, j, th, pol = best; e = np.clip(e, 1e-10, 1 - 1e-10); a = 0.5 * np.log((1 - e) / e)
        h = pol * np.where(X[:, j] > th, 1, -1); w = w * np.exp(-a * y * h); w /= w.sum(); out.append((a, j, th, pol))
    return out

def ada_predict(model, X):
    return (sum(a * p * np.where(X[:, j] > th, 1, -1) for a, j, th, p in model) > 0).astype(int)

rng3 = np.random.default_rng(3)
Xn, yn = make_ring(500, 0.0, rng3); fl = rng3.random(500) < 0.15; yn = np.where(fl, 1 - yn, yn)
ada = ada_stump_boost(Xn, yn, 100)
gbn = GBClassifier(100, 0.1, 1).fit(Xn, yn)
print("15%% label noise, 100 stumps each -> clean test acc: AdaBoost %.3f | GB log-loss %.3f"
      % ((ada_predict(ada, Xct) == yct).mean(), (gbn.predict(Xct) == yct).mean()))

## Summary
* `F_0 = argmin_c sum L(y,c)`; each round fits a shallow tree to the pseudo-residuals `-dL/dF`, picks the leaf step
  `gamma` (line search / Newton), and updates `F += eta * gamma * h`.
* Squared loss: pseudo-residual = residual. Log-loss: `y - sigmoid(F)`, leaf = `sum(r)/sum(p(1-p))`.
* Shrinkage `eta` + many trees + early stopping is the main regulariser; depth and subsample control variance.
* Choosing the loss (absolute, Huber, log) is the flexibility AdaBoost's fixed exponential loss lacks.